# 03 — Cohort Analysis (Customer Retention)

**Goal:** Group customers into monthly cohorts by their first purchase month, then track what fraction of each cohort keeps buying in subsequent months. 

Uses `outputs/data/cleaned_customer.csv` from `01_data_cleaning.ipynb`.

In [4]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [5]:
df = pd.read_csv(r"D:\ProProjects\data-analytics-projects\ecom\outputs\data\cleaned_customer.csv", parse_dates=['invoicedate'])
df.head()

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,iscancellation,totalprice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,False,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,False,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,20.34


## 1. Assign each transaction to an order month and each customer to a cohort month 

- `OrderMonth` — the month a given transaction happened
- `CohortMonth` — the month of that customer's very first transaction (their acquisition month)

In [6]:
df['ordermonth'] = df['invoicedate'].dt.to_period('M')
cohort_month = df.groupby('customerid')['ordermonth'].transform('min')
df['cohortmonth'] = cohort_month

df[['customerid', 'invoicedate', 'ordermonth', 'cohortmonth']].head()

,customerid,invoicedate,ordermonth,cohortmonth
0,17850,2010-12-01 08:26:00,2010-12,2010-12
1,17850,2010-12-01 08:26:00,2010-12,2010-12
2,17850,2010-12-01 08:26:00,2010-12,2010-12
3,17850,2010-12-01 08:26:00,2010-12,2010-12
4,17850,2010-12-01 08:26:00,2010-12,2010-12


In [7]:
# Cohort index = how many months after acquisition this order happened (0 = acquisition month itself)

def period_diff(order_month, cohort_month):
    return (order_month.year - cohort_month.year) * 12 + (order_month.month - cohort_month.month)

df['cohortindex'] = df.apply(lambda r: period_diff(r['ordermonth'], r['cohortmonth']), axis=1)
df[['customerid', 'cohortmonth', 'ordermonth', 'cohortindex']].head()

,customerid,cohortmonth,ordermonth,cohortindex
0,17850,2010-12,2010-12,0
1,17850,2010-12,2010-12,0
2,17850,2010-12,2010-12,0
3,17850,2010-12,2010-12,0
4,17850,2010-12,2010-12,0
